In [0]:
dbutils.widgets.removeAll()

In [0]:
# Create widget for ProcessedJSON parameter
dbutils.widgets.text("ProcessedJSON", "", "")
widget_value = dbutils.widgets.get("ProcessedJSON")

if widget_value:
    ProcessedJSON = widget_value

In [0]:
# Define notebook paths
procNotebook = "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/Shared/Notebooks/DatalakeProcessing/MoveFileToProcess"

In [0]:
from pyspark.sql.functions import explode, col
import json

In [0]:
%run "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/Shared/Notebooks/MoveFileToProcess"

In [0]:
%run "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/Shared/CommonMethods/Helpers/SynJSONCreatorClass"

In [0]:
%run "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/Shared/CommonMethods/Helpers/FileHandling"

In [0]:
if isinstance(ProcessedJSON, str):
    if not ProcessedJSON or ProcessedJSON.strip() == "":
        raise ValueError("ProcessedJSON parameter is required")
    ProcessedJSON = json.loads(ProcessedJSON)
elif not ProcessedJSON:
    raise ValueError("ProcessedJSON parameter is required")

filesDF = spark.createDataFrame([ProcessedJSON])

explodedFileIDs = filesDF.select(explode(col("FileIds"))).select(
    col("col.ClientID"),
    col("col.FileID"),
    col("col.FileName"),
    col("col.ClientContainer"),
    col("col.CurrentFolderPath"),
    col("col.ProcessedFolderPath"),
    col("col.ColumnDelimiter"),
    col("col.HasHeader"),
    col("col.IgnoreHeader"),
    col("col.FileLayoutID"),
    col("col.FileLayoutDescription"),
    col("col.SchemaFileName"),
    col("col.SchemaFilePath"),
    col("col.TextQualifier")
)

ErrorMessage = ""
doubleQuote = '"'

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
try:
    currentJobId = ctx.tags().get("jobId").getOrElse(lambda: "Undefined")
except Exception:
    currentJobId = "Undefined"

rJSON = synJSONCreator()
rJSON.addBracketStart()

iterator = 0
exploded_pd = explodedFileIDs.toPandas()

if exploded_pd.empty:
    raise ValueError("No files found in explodedFileIDs. Please check the input JSON.")

for index, row in exploded_pd.iterrows():
    print(f"Begin {row['FileID']}-{row['FileName']}")
    
    CurrPath = f"{row['ClientContainer']}{row['CurrentFolderPath']}"
    ProcessedPath = row['ProcessedFolderPath']
    SchemaFile = f"{row['SchemaFilePath']}/{row['SchemaFileName']}"
    FullFileName = f"{row['ClientContainer']}{row['CurrentFolderPath']}/{row['FileName']}"

    print(f"--> Target Data Path: file:{FullFileName}")
    print(f"--> Target Schema Path: file:{SchemaFile}")
    print(f"--> Target Output Path: {ProcessedPath}")

    # Validate Data File Presence
    try:
        check_path = FullFileName if FullFileName.startswith('/Volumes/') else f"file:{FullFileName}"
        dbutils.fs.ls(check_path)
        is_data_file_valid = True
    except Exception as e:
        print(f"Data file check failed: {str(e)}")
        is_data_file_valid = False

    # Validate Schema File Presence
    try:
        check_schema = SchemaFile if SchemaFile.startswith('/Workspace/') else f"file:{SchemaFile}"
        dbutils.fs.ls(check_schema)
        is_schema_file_valid = True
    except Exception as e:
        print(f"Schema file check failed: {str(e)}")
        is_schema_file_valid = False

    # Route Processing Based on Validation State
    if is_data_file_valid and is_schema_file_valid:
        if iterator != 0:
            rJSON.addComma()

        rJSON.addBraceStart()
        rJSON.addNewEntry("FileID", row['FileID'])
        rJSON.addNewEntry("FileName", row['FileName'])
        
        try:
            results = process_move_file(
                ClientId=row['ClientID'],
                FileId=row['FileID'],
                FileLayoutId=row['FileLayoutID'],
                FileLayoutDescription=row['FileLayoutDescription'],
                ColumnDelimiter=row['ColumnDelimiter'],
                HasHeader=row['HasHeader'],
                IgnoreHeader=row['IgnoreHeader'],
                textQualifier=row['TextQualifier'],
                FullFileName=FullFileName,
                SchemaFile=SchemaFile,
                ProcessedPath=ProcessedPath
            )

            returnedJson = spark.createDataFrame([json.loads(str(results))])

            for x in returnedJson.collect():
                rJSON.addNewEntry("FullFilePath", ProcessedPath)
                rJSON.addNewEntry("CurrentJobId", x['CurrentJobId'])
                rJSON.addNewEntry("Status", x['Status']) 
                rJSON.addNewEntry("RecordCount", x['ProcessedCount'])     
                rJSON.addNewEntry("ErrorMessage", x['ErrorMessage'], False)                                    
                
        except Exception as e:
            clean_err = str(e).strip().replace(doubleQuote, "")
            rJSON.addNewEntry("CurrentJobId", "Undefined")
            rJSON.addNewEntry("Status", "FAILURE")
            rJSON.addNewEntry("RecordCount", "")   
            rJSON.addNewEntry("ErrorMessage", clean_err, False)   
  
        rJSON.addBraceEnd()
  
    elif not is_data_file_valid:
        if iterator != 0: rJSON.addComma()
        rJSON.addBraceStart()
        rJSON.addNewEntry("CurrentJobId", "Undefined")
        rJSON.addNewEntry("FileID", row['FileID'])
        rJSON.addNewEntry("FileName", row['FileName'])
        rJSON.addNewEntry("FullFilePath", CurrPath)
        rJSON.addNewEntry("Status", "FAILED")
        rJSON.addNewEntry("RecordCount", "")   
        rJSON.addNewEntry("ErrorMessage", "Data File Not Found", False)
        rJSON.addBraceEnd()
        
    elif not is_schema_file_valid:
        if iterator != 0: rJSON.addComma()
        rJSON.addBraceStart()
        rJSON.addNewEntry("CurrentJobId", "Undefined")
        rJSON.addNewEntry("FileID", row['FileID'])
        rJSON.addNewEntry("FileName", row['SchemaFileName'])
        rJSON.addNewEntry("FullFilePath", CurrPath)
        rJSON.addNewEntry("Status", "FAILED")
        rJSON.addNewEntry("RecordCount", "")   
        rJSON.addNewEntry("ErrorMessage", "Schema File Not Found", False)
        rJSON.addBraceEnd()

    iterator += 1

rJSON.addBracketEnd()

returnVal = rJSON.getJSON()
print(returnVal)
dbutils.notebook.exit(returnVal)

In [0]:
# single file handling
# Convert to a single record dictionary instead of a looping DataFrame
if explodedFileIDs.count() == 0:
    raise ValueError("No files found in explodedFileIDs. Please check the input JSON.")

# Extract the single payload record directly
row = explodedFileIDs.first().asDict()

print(f"Begin {row['FileID']}-{row['FileName']}")

CurrPath = f"{row['ClientContainer']}{row['CurrentFolderPath']}"
ProcessedPath = row['ProcessedFolderPath']
SchemaFile = f"{row['SchemaFilePath']}/{row['SchemaFileName']}"
FullFileName = f"{row['ClientContainer']}{row['CurrentFolderPath']}/{row['FileName']}"

print(f"--> Target Data Path: file:{FullFileName}")
print(f"--> Target Schema Path: file:{SchemaFile}")
print(f"--> Target Output Path: {ProcessedPath}")

# Validate Data File Presence
try:
    check_path = FullFileName if FullFileName.startswith('/Volumes/') else f"file:{FullFileName}"
    dbutils.fs.ls(check_path)
    is_data_file_valid = True
except Exception as e:
    print(f"Data file check failed: {str(e)}")
    is_data_file_valid = False

# Validate Schema File Presence
try:
    check_schema = SchemaFile if SchemaFile.startswith('/Workspace/') else f"file:{SchemaFile}"
    dbutils.fs.ls(check_schema)
    is_schema_file_valid = True
except Exception as e:
    print(f"Schema file check failed: {str(e)}")
    is_schema_file_valid = False

# Process Single File Based on Validation State
if is_data_file_valid and is_schema_file_valid:
    rJSON.addBraceStart()
    rJSON.addNewEntry("FileID", row['FileID'])
    rJSON.addNewEntry("FileName", row['FileName'])
    
    try:
        results = process_move_file(
            ClientId=row['ClientID'],
            FileId=row['FileID'],
            FileLayoutId=row['FileLayoutID'],
            FileLayoutDescription=row['FileLayoutDescription'],
            ColumnDelimiter=row['ColumnDelimiter'],
            HasHeader=row['HasHeader'],
            IgnoreHeader=row['IgnoreHeader'],
            textQualifier=row['TextQualifier'],
            FullFileName=FullFileName,
            SchemaFile=SchemaFile,
            ProcessedPath=ProcessedPath
        )

        returnedJson = spark.createDataFrame([json.loads(str(results))])

        for x in returnedJson.collect():
            rJSON.addNewEntry("FullFilePath", ProcessedPath)
            rJSON.addNewEntry("CurrentJobId", x['CurrentJobId'])
            rJSON.addNewEntry("Status", x['Status']) 
            rJSON.addNewEntry("RecordCount", x['ProcessedCount'])     
            rJSON.addNewEntry("ErrorMessage", x['ErrorMessage'], False)                                    
            
    except Exception as e:
        clean_err = str(e).strip().replace(doubleQuote, "")
        rJSON.addNewEntry("CurrentJobId", "Undefined")
        rJSON.addNewEntry("Status", "FAILURE")
        rJSON.addNewEntry("RecordCount", "")   
        rJSON.addNewEntry("ErrorMessage", clean_err, False)   

    rJSON.addBraceEnd()

elif not is_data_file_valid:
    rJSON.addBraceStart()
    rJSON.addNewEntry("CurrentJobId", "Undefined")
    rJSON.addNewEntry("FileID", row['FileID'])
    rJSON.addNewEntry("FileName", row['FileName'])
    rJSON.addNewEntry("FullFilePath", CurrPath)
    rJSON.addNewEntry("Status", "FAILED")
    rJSON.addNewEntry("RecordCount", "")   
    rJSON.addNewEntry("ErrorMessage", "Data File Not Found", False)
    rJSON.addBraceEnd()
    
elif not is_schema_file_valid:
    rJSON.addBraceStart()
    rJSON.addNewEntry("CurrentJobId", "Undefined")
    rJSON.addNewEntry("FileID", row['FileID'])
    rJSON.addNewEntry("FileName", row['SchemaFileName'])
    rJSON.addNewEntry("FullFilePath", CurrPath)
    rJSON.addNewEntry("Status", "FAILED")
    rJSON.addNewEntry("RecordCount", "")   
    rJSON.addNewEntry("ErrorMessage", "Schema File Not Found", False)
    rJSON.addBraceEnd()